# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace
from udaplay.agent import Agent
import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"
print("config.py loads this to centralize configuration management. I'll refrain from doing this here.")

config.py loads this to centralize configuration management. I'll refrain from doing this here.


In [4]:
print("config.py loads this to centralize configuration management. I'll refrain from doing this here.")
# load_dotenv()

config.py loads this to centralize configuration management. I'll refrain from doing this here.


### VectorDB Instance

In [5]:
# Choose any path you want
# chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [7]:
# TODO: Create a collection
# Choose any name you want
# collection = chroma_client.create_collection(
#    name="udaplay",
#    embedding_function=embedding_fn
#)

### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
from udaplay.dataset import build_dataset
from udaplay.index_builder import build_all

print(build_dataset(target_count=1000), " games in data.")
print("We will be building 3 indexes to be able to compare the a baseline without chunking strategy, one with RAPTOR, and one with RAPTOR and SAC.")
print(build_all())

[dataset] 1000 game(s) in data/games.
1000  games in data.
We will be building 3 indexes to be able to compare the a baseline without chunking strategy, one with RAPTOR, and one with RAPTOR and SAC.


/Users/joranbergfeld/vcs/udacity/udacity-udaplay-research-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/joranbergfeld/vcs/udacity/udacity-udaplay-research-agent/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/joranbergfeld/vcs/udacity/udacity-udaplay-research-agent/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/joranbergfeld/vcs/udacity/udacity-udaplay-research-agent/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/joranbergfeld/

{'udaplay_baseline': 1000, 'udaplay_raptor': 1192, 'udaplay_raptor_sac': 1157}


In [17]:
from udaplay.index_builder import INDEX_BASELINE, INDEX_RAPTOR, INDEX_RAPTOR_SAC, get_collection

indexes = [INDEX_BASELINE, INDEX_RAPTOR, INDEX_RAPTOR_SAC]
print("Let's check the collection to make sure everything is in order.")
for index in indexes:
    collection = get_collection(index)
    print(f"Collection {index} has {collection.count()} documents.")

query = "realistic racing simulator on PlayStation"

for index in indexes:
    coll = get_collection(index)
    res = coll.query(query_texts=[query], n_results=50)   # fetch extra to survive filtering
    games = [(m["Name"], 1 - d)
             for m, d in zip(res["metadatas"][0], res["distances"][0]) if m.get("Name")]

    print(f"\n=== {index} ===")
    for rank, (name, score) in enumerate(games[:10], start=1):
        print(f"{rank}. {name}  (score={score:.3f})")

Let's check the collection to make sure everything is in order.
Collection udaplay_baseline has 1000 documents.
Collection udaplay_raptor has 1192 documents.
Collection udaplay_raptor_sac has 1157 documents.

=== udaplay_baseline ===
1. Gran Turismo  (score=0.895)
2. Gran Turismo 5  (score=0.888)
3. Hot Wheels Unleashed  (score=0.837)
4. Gran Turismo 7  (score=0.837)
5. Project CARS 3  (score=0.836)
6. Pacific Drive  (score=0.833)
7. F1 22  (score=0.832)
8. Destruction AllStars  (score=0.826)
9. Hotshot Racing  (score=0.824)
10. GRID Legends  (score=0.822)

=== udaplay_raptor ===
1. Gran Turismo  (score=0.895)
2. Gran Turismo 5  (score=0.888)
3. Gran Turismo 7  (score=0.837)
4. Hot Wheels Unleashed  (score=0.837)
5. Project CARS 3  (score=0.836)
6. Pacific Drive  (score=0.833)
7. F1 22  (score=0.832)
8. Destruction AllStars  (score=0.826)
9. Hotshot Racing  (score=0.824)
10. GRID Legends  (score=0.822)

=== udaplay_raptor_sac ===
1. Gran Turismo  (score=0.879)
2. Gran Turismo 5  (score